# Alignment Debug Notebook

This notebook provides visual verification of the alignment results.
Use this to quickly check that time deltas are reasonable and alignment is working correctly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.style.use('default')  # Use matplotlib defaults (colorblind-friendly)

## Load Aligned Data

In [ ]:
# Specify the aligned file to analyze
aligned_file = Path('aligned_data/007_Fast_stbd_turn_1_aligned.h5')

if not aligned_file.exists():
    print(f"File not found: {aligned_file}")
    print("Available files:")
    for f in Path('aligned_data').glob('*.h5'):
        print(f"  {f.name}")
else:
    # Load the HDF5 store
    store = pd.HDFStore(str(aligned_file), mode='r')
    print(f"Loaded: {aligned_file}")
    print(f"\nAvailable sensors:")
    for key in store.keys():
        if key != '/metadata':
            print(f"  {key}: {len(store[key])} samples")

## Check Alignment Metadata

In [ ]:
# Load metadata
metadata = store['metadata']
print("Alignment Metadata:")
print(metadata.T)

## Visualize Time Deltas

In [ ]:
# Plot time differences for each aligned sensor
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

sensor_idx = 0
for sensor_name in ['sensor_3', 'sensor_4', 'sensor_5', 'sensor_wb']:
    if f'/{sensor_name}' in store:
        df = store[sensor_name]
        
        if 'time_diff_ms' in df.columns:
            ax = axes[sensor_idx]
            
            # Plot histogram of time differences
            ax.hist(df['time_diff_ms'], bins=50, alpha=0.7, edgecolor='black')
            ax.set_xlabel('Time Difference (ms)')
            ax.set_ylabel('Count')
            ax.set_title(f'{sensor_name} Time Deltas')
            
            # Add statistics
            mean_diff = df['time_diff_ms'].mean()
            max_diff = df['time_diff_ms'].max()
            ax.text(0.95, 0.95, f'Mean: {mean_diff:.3f} ms\nMax: {max_diff:.3f} ms',
                   transform=ax.transAxes, ha='right', va='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            sensor_idx += 1

# Hide unused subplots
for i in range(sensor_idx, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.suptitle('Time Alignment Quality Check', y=1.02)
plt.show()

## Check Alignment Consistency Over Time

In [ ]:
# Plot time differences over the duration of the experiment
plt.figure(figsize=(14, 6))

for sensor_name in ['sensor_4', 'sensor_5', 'sensor_wb']:
    if f'/{sensor_name}' in store:
        df = store[sensor_name]
        
        if 'time_diff_ms' in df.columns and 'aligned_time' in df.columns:
            # Sample every 100th point to avoid overplotting
            sample_idx = slice(None, None, 100)
            plt.plot(df['aligned_time'].iloc[sample_idx], 
                    df['time_diff_ms'].iloc[sample_idx],
                    'o', markersize=2, alpha=0.6, label=sensor_name)

plt.xlabel('Aligned Time (s)')
plt.ylabel('Time Difference (ms)')
plt.title('Alignment Time Differences Throughout Experiment')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Sample Rate Verification

In [ ]:
# Check actual sample rates after alignment
print("Effective Sample Rates After Alignment:\n")

for sensor_name in store.keys():
    if sensor_name != '/metadata':
        df = store[sensor_name]
        
        if 'aligned_time' in df.columns and len(df) > 1:
            # Calculate time differences between consecutive samples
            time_diffs = np.diff(df['aligned_time'].values)
            
            # Filter out any large gaps
            valid_diffs = time_diffs[time_diffs < 0.1]  # Ignore gaps > 100ms
            
            if len(valid_diffs) > 0:
                mean_period = np.mean(valid_diffs)
                effective_rate = 1.0 / mean_period
                
                print(f"{sensor_name[1:]:12s}: {effective_rate:6.1f} Hz (period: {mean_period*1000:.2f} ms)")

## Close the HDF5 Store

In [ ]:
# Always close the store when done
if 'store' in locals():
    store.close()
    print("HDF5 store closed.")